In [ ]:
import torch 
import torch.nn as nn 
import torch.nn.functional as F 
import torch.optim as optim
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import torchvision 
from torchvision import datasets 
import random

In [ ]:
seed = 1234 
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

In [ ]:
import os 
train_video_dir = "C:/Users/araya/Desktop/vscode/Deep-Learning-Coding/MediaPipe/Data for Different actions/train"
validate_video_dir = "C:/Users/araya/Desktop/vscode/Deep-Learning-Coding/MediaPipe/Data for Different actions/validate"
test_video_dir = "C:/Users/araya/Desktop/vscode/Deep-Learning-Coding/MediaPipe/Data for Different actions/test"
train_video_list = []
validate_video_list = []
test_video_list = []
train_video_list = os.listdir(train_video_dir)
validate_video_list = os.listdir(validate_video_dir)
test_video_list = os.listdir(test_video_dir)

print("Number of train_video_list : ", len(train_video_list))
print("List of train_video_list : ", train_video_list)
print("Number of validate_video_list : ", len(validate_video_list))
print("List of validate_video_list : ", validate_video_list)
print("Number of test_video_list : ", len(test_video_list))
print("List of test_video_list : ", test_video_list)

In [ ]:
train_actions = np.array(train_video_list)
print(train_actions)
validate_actions = np.array(validate_video_list)
print(validate_actions)
test_actions = np.array(test_video_list)
print(test_actions)

In [ ]:
train_file_paths = []
for action in train_actions:
    video_path = os.path.join('Data for different actions/train/', action)
    # print(video_path)
    # print(action)
    train_file_paths.append(video_path + '/' + action.split(".")[0] + ".npy")
print(train_file_paths)

In [ ]:
validate_file_paths = []
for action in validate_actions:
    video_path = os.path.join('Data for different actions/validate/', action)
    # print(video_path)
    # print(action)
    validate_file_paths.append(video_path + '/' + action.split(".")[0] + ".npy")
print(validate_file_paths)

In [ ]:
test_file_paths = []
for action in test_actions:
    video_path = os.path.join('Data for different actions/test/', action)
    # print(video_path)
    # print(action)
    test_file_paths.append(video_path + '/' + action.split(".")[0] + ".npy")
print(test_file_paths)

In [ ]:
import pandas as pd
train_df = pd.read_csv("script4.csv")
validate_df = pd.read_csv("script4_validate.csv")
test_df = pd.read_csv("script4_test.csv")

In [ ]:
train_labels = []
validate_labels = []
test_labels = []

for i in range(len(train_df.label)):
    train_labels.append(train_df.label[i])

for i in range(len(validate_df.label)):
    validate_labels.append(validate_df.label[i])

for i in range(len(test_df.label)):
    test_labels.append(test_df.label[i])

In [ ]:
print("Train Label : ", train_labels)
print("Validate Label : ",validate_labels)
print("Test Label : ", test_labels)

In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence

# Create a custom dataset
class KeypointDataset(Dataset):
    def __init__(self, file_paths, labels):
        self.file_paths = file_paths
        self.labels = labels
    
    def __len__(self):
        return len(self.file_paths)
    
    def __getitem__(self, idx):
        keypoints = np.load(self.file_paths[idx])
        label = self.labels[idx]
        return torch.tensor(keypoints, dtype=torch.float32), label

In [ ]:
train_dataset = KeypointDataset(train_file_paths, train_labels)
validate_dataset = KeypointDataset(validate_file_paths, validate_labels)
test_dataset = KeypointDataset(test_file_paths, test_labels)

In [ ]:
# Collate function for padding
def collate_fn(batch):
    sequences, labels = zip(*batch)
    padded_sequences = pad_sequence(sequences, batch_first=True)
    return padded_sequences, torch.tensor(labels)

In [ ]:
# Create the DataLoader
batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
val_loader = DataLoader(validate_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

In [ ]:
for i, (data, label) in enumerate(train_loader):
    print(data)
    print(label)
    break

In [ ]:
print(data.shape)
print(label.shape)

### Model

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-torch.log(torch.tensor(10000.0)) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:x.size(0), :]
        return x

In [ ]:
positional = PositionalEncoding(d_model=1086)

In [ ]:
position = positional(data)
print(position)

In [ ]:
print(data)

In [ ]:
print(data.shape)
print(position.shape)

In [ ]:
import math

class PositionalEncodingSinCos(nn.Module):

    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, 1, d_model)
        pe[:, 0, 0::2] = torch.sin(position * div_term)
        pe[:, 0, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Tensor, shape [seq_len, batch_size, embedding_dim]
        """
        posEnc = self.pe[:x.size(0)]
        xAndPosEnc = x + posEnc
        return self.dropout(xAndPosEnc)

In [ ]:
positionalencodingsincos = PositionalEncodingSinCos(d_model=1086)

In [ ]:
res = positionalencodingsincos(data)
print(res)

In [ ]:
print(data.shape)
print(res.shape)

In [ ]:
class BaselineTransformerClassification(nn.Module):
    """
    Implementation of the SPOTER (Sign POse-based TransformER) architecture for sign language recognition from sequence
    of skeletal data.
    """

    def __init__(self, num_classes, hidden_dim, n_heads, max_seq_len = 5000):
        super().__init__()

        self.output_pos_encoding = PositionalEncoding(hidden_dim)

        self.transformer = nn.Transformer(hidden_dim, n_heads, 6, 6)
        self.linear_class = nn.Linear(hidden_dim, num_classes)


    def forward(self, inputs):

        # h = torch.unsqueeze(inputs.flatten(start_dim=1), 1).float()
        #Apply positional emb.
        henc = self.output_pos_encoding(inputs)


        h = self.transformer(henc, henc)
        # Avg. Pooling in timesteps
        pooled = torch.mean(h, dim=0)
        res = self.linear_class(pooled)

        return res

In [ ]:
input_dim = 1086
# d_model = 512
n_heads = 6
hidden_dim = 72
n_layers = 1
num_classes = 32
dropout = 0.5

In [ ]:
model = BaselineTransformerClassification(num_classes=num_classes, n_heads=n_heads, hidden_dim=hidden_dim).to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()  # For multi-class classification 
optimizer = optim.Adam(model.parameters(), lr=0.0001)

In [ ]:
from sklearn.metrics import f1_score, recall_score

num_epochs = 100
loss_logger = []
accuracy_logger = []
f1_logger = []
recall_logger = []
running_loss = 0.0 
val_loss = []

all_preds = []
all_labels = []
for epoch in range(num_epochs):
    model.train()
    total_correct = 0
    total_samples = 0
    
    all_preds = []
    all_labels = []

    for i, (sequences, labels) in enumerate(train_loader):
        sequences = sequences.to(device)
        labels = labels.to(device)
        
        # Forward pass
        outputs = model(sequences)
        _, predicted = torch.max(outputs, 1)
        # outputs = outputs.sum(dim=0)
        loss = criterion(outputs, labels)

        total_correct += (predicted == labels).sum().item()
        total_samples += labels.size(0)

        all_preds.extend(predicted.cpu().numpy())  # Store predictions
        all_labels.extend(labels.cpu().numpy())    # Store true labels
        
        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # Log training loss and accuracy
    loss_logger.append(loss.item())
    accuracy = 100 * total_correct / total_samples
    accuracy_logger.append(accuracy)

    # Calculate F1 score and recall for training set
    f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=1)
    recall = recall_score(all_labels, all_preds, average='weighted', zero_division=1)
    f1_logger.append(f1)
    recall_logger.append(recall)

    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}, Accuracy: {accuracy:.2f}%, F1 Score: {f1:.2f}, Recall: {recall:.2f}')

    # --- Validation Loop ---
    model.eval()  # Set the model to evaluation mode
    val_correct = 0
    val_samples = 0
    val_preds = []
    val_labels = []
    running_val_loss = 0.0  # Reset running loss for validation

    with torch.no_grad():  # Disable gradient calculation for validation

        for sequences, labels in val_loader:
            sequences = sequences.to(device)
            labels = labels.to(device)

            # Forward pass
            outputs = model(sequences)
            loss = criterion(outputs, labels) 
            running_val_loss += loss.item() 

            _, predicted = torch.max(outputs, 1)
            val_correct += (predicted == labels).sum().item()
            val_samples += labels.size(0)

            val_preds.extend(predicted.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())

        avg_val_loss = running_val_loss / len(val_loader)
        val_loss.append(avg_val_loss)

        val_accuracy = 100 * val_correct / val_samples
        val_f1 = f1_score(val_labels, val_preds, average='weighted', zero_division=1 )
        val_recall = recall_score(val_labels, val_preds, average='weighted', zero_division=1)

        print(f'Validation Loss: {avg_val_loss:.4f}, Accuracy: {val_accuracy:.2f}%, F1 Score: {val_f1:.2f}, Recall: {val_recall:.2f}')

# --- Test Loop (after all epochs) ---
model.eval()  # Set model to evaluation mode for testing
test_correct = 0
test_samples = 0
test_preds = []
test_labels = []

with torch.no_grad():  # Disable gradient calculation for testing

    for sequences, labels in test_loader:
        sequences = sequences.to(device)
        labels = labels.to(device)

        # Forward pass
        outputs = model(sequences)
        _, predicted = torch.max(outputs, 1)

        test_correct += (predicted == labels).sum().item()
        test_samples += labels.size(0)

        test_preds.extend(predicted.cpu().numpy())
        test_labels.extend(labels.cpu().numpy())

    test_accuracy = 100 * test_correct / test_samples
    test_f1 = f1_score(test_labels, test_preds, average='weighted', zero_division=1 )
    test_recall = recall_score(test_labels, test_preds, average='weighted', zero_division=1)

    print(f'Test Accuracy: {test_accuracy:.2f}%, F1 Score: {test_f1:.2f}, Recall: {test_recall:.2f}')
